# 04 — Live Demo: end-to-end на 3 основных доменах

Краткий ноутбук для **демонстрации на защите**: 3 SKU (pasta / chocolate / cheeses) проходят через каскадный конвейер; виден trace «какой слой что предсказал».

Структурно дополняет монолит `00_thesis_main.ipynb` (§3.3 диссертации в исполняемом виде). Этот ноутбук **не** дублирует §3.3 — это короткий showcase для live-runs.

**На защите:** outputs закоммичены — если запуск на стенде упадёт, jury видит сохранённые результаты.

**Зависимости:** работающий стек `demo/ml_service/cascade.py` + ML-модели в `models/`, регенерируемые `reproduce.sh`.

## 1. Подготовка окружения

In [1]:
%env OMP_NUM_THREADS=1
import sys
assert sys.version_info[:2] in [(3, 12), (3, 14)], f"Unsupported Python {sys.version_info[:2]}"
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
_ml_svc = PROJECT_ROOT / 'demo' / 'ml_service'
if str(_ml_svc) not in sys.path:
    sys.path.insert(0, str(_ml_svc))

try:
    from demo.ml_service.cascade import CascadePipeline
    pipeline = CascadePipeline()
    print('CascadePipeline загружен — модели готовы')
except Exception as e:
    pipeline = None
    print(f'CascadePipeline не загружен: {type(e).__name__}: {e}')

env: OMP_NUM_THREADS=1


Validator artifacts missing for pasta_v4


Validator artifacts missing for chocolate_v4


Validator artifacts missing for cheeses_v4


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CategoryRouter artefacts not found: no router artefact found in /Users/miafrolov/Desktop/stuff/ai_attributes/models — train with `python -m src.pipeline.category_router.train` (v1) or `python -m src.pipeline.category_router.train_with_adversarial` (v4) — auto mode disabled


MahalanobisOOD artefact missing: missing Mahalanobis artefact: /Users/miafrolov/Desktop/stuff/ai_attributes/models/category_router_mahalanobis.npz — train with `python -m src.pipeline.category_router.fit_mahalanobis` — semantic OOD disabled


CascadePipeline загружен — модели готовы


## 2. Утилита pretty-print: показать какой слой что предсказал

In [2]:
def show_result(category: str, product: dict, result: dict) -> None:
    print('=' * 80)
    print(f'КАТЕГОРИЯ: {category}')
    print(f'ВХОД: {product["product_name"]}' + (f' [{product["brands"]}]' if product.get('brands') else ''))
    if product.get('quantity'):
        print(f'      quantity: {product["quantity"]}')
    if product.get('ingredients_text'):
        ing = product['ingredients_text']
        print(f'      ingredients: {ing[:80]}' + ('…' if len(ing) > 80 else ''))
    print('-' * 80)
    predictions = result.get('predictions', {})
    if not predictions:
        print('(нет предсказаний)')
        return
    print(f'{"АТРИБУТ":<28} | {"ЗНАЧЕНИЕ":<22} | {"СЛОЙ":<10} | УВЕРЕННОСТЬ')
    print('-' * 80)
    for attr, block in predictions.items():
        value = str(block.get('value', '—'))[:20]
        layer = block.get('layer', '—')
        conf = block.get('confidence', 0.0)
        validated = block.get('validation') or {}
        flag = ' ⚠FLAG' if validated.get('flagged') else ''
        print(f'{attr:<28} | {value:<22} | {layer:<10} | {conf:.2f}{flag}')
    n_total = result.get('n_attrs_total', 0)
    n_covered = result.get('n_covered', 0)
    n_llm = result.get('n_llm_fallback', 0)
    print('-' * 80)
    print(f'Покрыто: {n_covered}/{n_total} атрибутов  |  LLM-fallback: {n_llm}')
    print()

## 3. SKU №1 — Pasta (flagship-кейс с полным циклом аудита)

In [3]:
pasta_product = {
    'product_name': 'De Cecco penne rigate n.41',
    'brands': 'De Cecco',
    'ingredients_text': 'Semoule de blé dur de qualité supérieure',
    'quantity': '500 g',
    'code': '8001250200419',
}
try:
    pasta_result = pipeline.predict('pasta', pasta_product) if pipeline else {}
    show_result('pasta', pasta_product, pasta_result)
except Exception as e:
    pasta_result = {}
    print(f'[pasta] predict failed: {type(e).__name__}: {e}')

[pasta] predict failed: ValueError: Feature shape mismatch, expected: 384, got 768


## 4. SKU №2 — Chocolate

In [4]:
chocolate_product = {
    'product_name': 'Lindt Excellence 70 % Cocoa Dark Chocolate',
    'brands': 'Lindt',
    'ingredients_text': 'cocoa mass, sugar, cocoa butter, demerara sugar, bourbon vanilla beans',
    'quantity': '100 g',
    'code': '3046920022606',
}
try:
    chocolate_result = pipeline.predict('chocolate', chocolate_product) if pipeline else {}
    show_result('chocolate', chocolate_product, chocolate_result)
except Exception as e:
    chocolate_result = {}
    print(f'[chocolate] predict failed: {type(e).__name__}: {e}')

[chocolate] predict failed: ValueError: Feature shape mismatch, expected: 384, got 768


## 5. SKU №3 — Cheeses (поздно добавленный домен с Layer-1 regex)

In [5]:
cheeses_product = {
    'product_name': 'Camembert de Normandie AOP',
    'brands': 'Président',
    'ingredients_text': 'lait cru de vache, sel, ferments lactiques, présure',
    'quantity': '250 g',
    'code': '3228020670064',
}
try:
    cheeses_result = pipeline.predict('cheeses', cheeses_product) if pipeline else {}
    show_result('cheeses', cheeses_product, cheeses_result)
except Exception as e:
    cheeses_result = {}
    print(f'[cheeses] predict failed: {type(e).__name__}: {e}')

[cheeses] predict failed: ValueError: Feature shape mismatch, expected: 384, got 768


## 6. Сводка по 3 SKU

In [6]:
from collections import Counter

print(f'{"КАТЕГОРИЯ":<14} | {"ПОКРЫТО":<10} | {"L1":<3} | {"L2":<3} | {"L3":<3} | {"L4":<3} | {"FLAGS":<6}')
print('-' * 70)
for label, res in [('pasta', pasta_result), ('chocolate', chocolate_result), ('cheeses', cheeses_result)]:
    preds = res.get('predictions', {})
    layers = Counter(p.get('layer', '?') for p in preds.values())
    flags = sum(1 for p in preds.values() if (p.get('validation') or {}).get('flagged'))
    covered = f'{res.get("n_covered", 0)}/{res.get("n_attrs_total", 0)}'
    print(f'{label:<14} | {covered:<10} | {layers.get("regex", 0):<3} | {layers.get("ml", 0):<3} | {layers.get("bayes", 0):<3} | {layers.get("llm", 0):<3} | {flags:<6}')
print()
print('Trace показывает, как каскад делегирует решения по слоям.')
print('L1 (regex) — числовые/паттерн-атрибуты; L2 (ML) — семантические; L3 (Bayes) — флаг подозрения; L4 (LLM) — запасной слой.')

КАТЕГОРИЯ      | ПОКРЫТО    | L1  | L2  | L3  | L4  | FLAGS 
----------------------------------------------------------------------
pasta          | 0/0        | 0   | 0   | 0   | 0   | 0     
chocolate      | 0/0        | 0   | 0   | 0   | 0   | 0     
cheeses        | 0/0        | 0   | 0   | 0   | 0   | 0     

Trace показывает, как каскад делегирует решения по слоям.
L1 (regex) — числовые/паттерн-атрибуты; L2 (ML) — семантические; L3 (Bayes) — флаг подозрения; L4 (LLM) — запасной слой.
